# jobs · 可复现分析 notebook

> BLS 342 个美国职业的 treemap + LLM 可换上色层

**完整报告**：[`jobs.html`](jobs.html)（31K / ~7.8K 字）

```bash
git clone https://github.com/karpathy/jobs ~/auto-research/jobs
```

## 执行摘要
- 11 commits / 2 天（2026-03-14 → 03-15）
- **最重要事件**：`1cca284` "update the readme to minimize misunderstanding"——从"AI 暴露分析"→"BLS 探索工具"的定位反转
- 核心技术精妙点：**AI Exposure 被定义为"工作产物是否原生数字"**，窄到可操作，避开"被替代概率"的预测雷
- 新兴设计模式：`prompt.md` 是"LLM-as-audience" artifact 的早期样例
- live demo: [karpathy.ai/jobs](https://karpathy.ai/jobs)

In [ ]:
import subprocess
from pathlib import Path
REPO = Path.home() / 'auto-research' / 'jobs'
def git(*args): return subprocess.check_output(['git', '-C', str(REPO), *args], text=True)
print('commits:', git('rev-list', '--count', 'HEAD').strip())

## 1. 分钟级时间线 — 28.5 小时沉默段

In [ ]:
log = git('log', '--reverse', '--date=format:%m-%d %H:%M', '--pretty=format:%h | %ad | %s')
print(log)

**观察**：3/14 15:39 → 3/15 20:10 存在 28.5 小时的沉默段。恢复后的第一批 commits 明确回应"读者误解"——`Redesign` + `update the readme to minimize misunderstanding`。这几乎可以肯定中间有外部反馈（Twitter / HN）推了作者一把。

## 2. 核心事件 — `1cca284` 定位反转

六处协同修改稀释 "AI Exposure" 主叙事，把项目从"一份分析"重定位为"一个探索工具"。

In [ ]:
print(git('show', '1cca284', '--stat').split('\n\n', 1)[0])

In [ ]:
# 标题 + 导语的反转证据
diff = git('show', '1cca284', '--format=', '--', 'README.md')
lines = diff.splitlines()
for l in lines[:50]:
    if l.startswith('-') or l.startswith('+'):
        print(l)

六处改动：
1. 标题 `"AI Exposure of the US Job Market" → "US Job Market"`
2. 导语加 `"This is not a report, a paper, or a serious economic publication"`
3. `what's here` 段改成"四个可切换 color layer"（AI 从主题降为 1/4）
4. 新增 `"LLM-powered coloring"` 整段（工具化立场）
5. 新增 `"What AI Exposure is NOT"` 免责段
6. 删除 "Average exposure 5.3/10" + calibration 表（移入 `score.py`）

**学到的**：Frey-Osborne 2013 踩过的雷（"47% of jobs at risk" 被媒体简化十年）——作者在 30 小时内就用同一条 commit 把这个雷主动拆了。

## 3. `score.py` 的 LLM rubric — 技术精妙点

"AI 暴露" 被定义为 "工作产物是否原生数字"——不是自动化概率、不是 displacement、不是 wage impact。这是**定义的克制**。

In [ ]:
# 读 score.py 的 SYSTEM_PROMPT
txt = (REPO / 'score.py').read_text()
# 切出 SYSTEM_PROMPT 块
start = txt.find('SYSTEM_PROMPT = """')
end = txt.find('"""\n', start + 20)
print(txt[start:end+3])

**读这段 rubric 的要点**：
- 只评一个维度（0-10），不做多轴
- 用具体职业锚定每个分数档（roofer 0-1 / RN 4-5 / software dev 8-9 / data entry 10）
- 明说"不是预测消失，是评估重塑程度"
- 强制 JSON 输出格式（机器可 parse）

**技术精妙**：窄定义让 LLM 打分有一致性（variance 更低），同时避开预测雷区。

## 4. `prompt.md` — "LLM-as-audience" 新模式

In [ ]:
print(git('show', '7fa0be6', '--stat').split('\n\n', 1)[0])
print('\n--- commit msg ---')
print(git('show', '-s', '--format=%B', '7fa0be6'))

In [ ]:
# prompt.md 的规模 — 给 LLM 消费的单文件聚合
prompt_path = REPO / 'prompt.md'
if prompt_path.exists():
    text = prompt_path.read_text()
    # 粗略 token 估算（4 字符 ≈ 1 token）
    print(f'prompt.md: {len(text)} 字符, 约 {len(text) // 4} tokens')
    print(f'\n前 50 行预览:')
    print('\n'.join(text.splitlines()[:50]))

**新兴设计模式**：repo 专门为 LLM 受众准备一份 artifact——不是给机器解析（JSON/CSV）、不是给人类全程读（文档），而是"粘贴到 LLM 对话里"用的聚合 Markdown 快照。

**对比**：
- 传统 `README.md` → 给人类读
- 传统 `data.json` → 给机器解析
- **新：`prompt.md` → 给 LLM 对话**

## 5. 6 阶段 pipeline

In [ ]:
# 识别 pipeline 各阶段脚本
scripts = ['scrape.py', 'parse_detail.py', 'process.py', 'make_csv.py',
           'score.py', 'build_site_data.py']
for s in scripts:
    p = REPO / s
    if p.exists():
        lines = len(p.read_text().splitlines())
        print(f'  {s:<24} {lines:>4} lines')

## 6. Takeaways

1. **定位反转比任何代码都重要**：`1cca284` 是"如何避开 Frey-Osborne 媒体化陷阱"的活教材。
2. **定义的克制是技术精妙点**：AI Exposure = "工作产物是否原生数字"。窄但可操作，避预测雷。
3. **LLM-as-audience** 是 2026 正在成型的一等设计模式——`prompt.md` 值得单独抄。
4. **发了就发了**：如果 30 小时内会有误解出现，就主动去改 framing——不要等人替你总结错。

完整分析见 [`jobs.html`](jobs.html)。